In [9]:
import pandas as pd
import numpy as np
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp
import matplotlib.pyplot as plt
import neurokit2 as nk
import spikeinterface.extractors as se
import glob
import os
import h5py
import numpy as np
from scipy.signal import butter, filtfilt, resample_poly
from scipy.signal import find_peaks
from collections import defaultdict
import math
import cv2
import matplotlib.animation as animation

In [17]:
import glob
import os

h5_root = r"D:\BLA_ChR_resp"

all_h5 = glob.glob(os.path.join(h5_root, "**", "*.h5"), recursive=True)

print("Total H5 files found:", len(all_h5))


Total H5 files found: 97


In [3]:
import os
os.path.exists(mp4_path)

True

In [4]:
h5_file = r"D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day1\h5_outputs\9_1_RI1_m2_m_on_20260111_165857.h5"
h5_off = r"D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day2\h5_outputs\1_1_RI1_f2_e_off_20260110_154830.h5"

In [7]:
def load_resp_simple(h5_file, target_rate=100, trodes_rate=20000):
    import h5py
    import numpy as np
    from scipy.signal import butter, filtfilt, resample_poly
    import neurokit2 as nk

    with h5py.File(h5_file, 'r') as f:
        resp_key = [k for k in f['analog'].keys() if "Ain1" in k][0]
        resp = f['analog'][resp_key][()]['voltage'].astype(float)

    # Time relative to recording start
    t = np.arange(len(resp)) / trodes_rate

    # Clean
    resp -= np.median(resp)
    mad = np.median(np.abs(resp))
    if mad > 0:
        resp = np.clip(resp, -10 * mad, 10 * mad)

    # Low-pass + downsample
    nyq = trodes_rate / 2
    b, a = butter(4, (target_rate / 2) / nyq, btype="low")
    resp_filt = filtfilt(b, a, resp)

    down = int(trodes_rate // target_rate)
    resp_ds = resample_poly(resp_filt, up=1, down=down)
    t_ds = t[::down]

    # Bandpass
    resp_clean = nk.signal_filter(
        resp_ds,
        lowcut=0.1,
        highcut=20,
        sampling_rate=target_rate,
        order=2
    )

    return resp_clean, t_ds

In [13]:
# ======================================================
# USER SETTINGS
# ======================================================

CLIP_START = 60
CLIP_END   = 240
WINDOW     = 5

rec_root = r"D:\BLA_ChR_resp"   # <-- root where .rec folders live
qc_folder = r"D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2"
os.makedirs(qc_folder, exist_ok=True)

# ======================================================
# AUTO-MATCH VIDEO TO H5
# ======================================================

# Extract base session name
base_name = os.path.basename(h5_file).replace(".h5", "")

print("Looking for video matching:", base_name)

# Find matching .rec folder
rec_matches = glob.glob(os.path.join(rec_root, "**", base_name + ".rec"), recursive=True)

if len(rec_matches) == 0:
    raise ValueError("No matching .rec folder found!")

rec_folder = rec_matches[0]
print("Matched REC folder:", rec_folder)

# Find mp4 inside .rec
mp4_matches = glob.glob(os.path.join(rec_folder, "*.mp4"))

if len(mp4_matches) == 0:
    raise ValueError("No mp4 found inside rec folder!")

mp4_path = mp4_matches[0]
print("Using MP4:", mp4_path)

# ======================================================
# LOAD VIDEO
# ======================================================

cap = cv2.VideoCapture(mp4_path)
fps = cap.get(cv2.CAP_PROP_FPS)

start_frame = int(CLIP_START * fps)
end_frame   = int(CLIP_END * fps)

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

# ======================================================
# LOAD RESPIRATION
# ======================================================

rsp, t = load_resp_simple(h5_file)

mask = (t >= CLIP_START) & (t <= CLIP_END)
t_clip = t[mask] - CLIP_START
rsp_clip = rsp[mask]

# ======================================================
# FIGURE SETUP
# ======================================================

fig, (ax_vid, ax_rsp) = plt.subplots(
    2, 1,
    figsize=(7, 7),
    gridspec_kw={"height_ratios": [3, 1]}
)

ret, frame = cap.read()
if not ret:
    raise RuntimeError("Could not read first frame.")

im = ax_vid.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
ax_vid.set_title(base_name)
ax_vid.axis("off")

line, = ax_rsp.plot([], [], color="black", lw=1)
ax_rsp.set_ylim(np.percentile(rsp_clip, 1), np.percentile(rsp_clip, 99))
ax_rsp.set_xlabel("Time (s)")
ax_rsp.set_ylabel("Respiration")

# ======================================================
# UPDATE FUNCTION
# ======================================================

def update(frame_idx):

    actual_frame = start_frame + frame_idx
    if actual_frame >= end_frame:
        return im, line

    cap.set(cv2.CAP_PROP_POS_FRAMES, actual_frame)
    ret, frame = cap.read()
    if not ret:
        return im, line

    im.set_data(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    current_time = frame_idx / fps

    window_mask = (
        (t_clip >= current_time - WINDOW) &
        (t_clip <= current_time)
    )

    line.set_data(t_clip[window_mask], rsp_clip[window_mask])
    ax_rsp.set_xlim(max(0, current_time - WINDOW), current_time)

    return im, line

# ======================================================
# CREATE ANIMATION
# ======================================================

ani = animation.FuncAnimation(
    fig,
    update,
    frames=(end_frame - start_frame),
    interval=1000 / fps,
    blit=False
)

# ======================================================
# SAVE
# ======================================================

clip_label = f"{CLIP_START}-{CLIP_END}s"
save_filename = f"{base_name}_{clip_label}_resp_overlay.mp4"
SAVE_PATH = os.path.join(qc_folder, save_filename)

print("Saving to:", SAVE_PATH)

ani.save(SAVE_PATH, writer="ffmpeg", fps=fps)

print("Saved successfully.")
plt.close(fig)


Looking for video matching: 9_1_RI1_m2_m_on_20260111_165857
Matched REC folder: D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day1\9_1_RI1_m2_m_on_20260111_165857.rec
Using MP4: D:\BLA_ChR_resp\BLA_ChR_resp_RI1_hc_day1\9_1_RI1_m2_m_on_20260111_165857.rec\9_1_RI1_m2_m_on_20260111_165857.1.mp4
Saving to: D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2\9_1_RI1_m2_m_on_20260111_165857_60-240s_resp_overlay.mp4


C:\Users\sjs93\AppData\Local\Temp\ipykernel_26140\2264882252.py:110: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax_rsp.set_xlim(max(0, current_time - WINDOW), current_time)


Saved successfully.


In [29]:
import glob
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# ======================================================
# USER SETTINGS
# ======================================================

CLIP_START = 60
CLIP_END   = 240
WINDOW     = 5

rec_root = r"D:\BLA_ChR_resp"
qc_folder = r"D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2"
os.makedirs(qc_folder, exist_ok=True)

# ======================================================
# FIND ALL H5 FILES
# ======================================================

all_h5 = glob.glob(os.path.join(rec_root, "**", "*.h5"), recursive=True)

print("Total H5 files found:", len(all_h5))

# ======================================================
# LOOP THROUGH EACH H5 FILE
# ======================================================

for h5_file in all_h5:

    base_name = os.path.basename(h5_file).replace(".h5", "")
    print("\n==============================")
    print("Processing:", base_name)
    print("==============================")

    # --------------------------------------------------
    # MATCH VIDEO
    # --------------------------------------------------

    rec_matches = glob.glob(
        os.path.join(rec_root, "**", base_name + ".rec"),
        recursive=True
    )

    if len(rec_matches) == 0:
        print("⚠ No matching .rec found. Skipping.")
        continue

    rec_folder = rec_matches[0]

    mp4_matches = glob.glob(os.path.join(rec_folder, "*.mp4"))

    if len(mp4_matches) == 0:
        print("⚠ No mp4 found in rec folder. Skipping.")
        continue

    mp4_path = mp4_matches[0]

    # --------------------------------------------------
    # LOAD VIDEO
    # --------------------------------------------------

    cap = cv2.VideoCapture(mp4_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    start_frame = int(CLIP_START * fps)
    end_frame   = int(CLIP_END * fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    # --------------------------------------------------
    # LOAD RESPIRATION
    # --------------------------------------------------

    rsp, t = load_resp_simple(h5_file)

    mask = (t >= CLIP_START) & (t <= CLIP_END)
    t_clip = t[mask] - CLIP_START
    rsp_clip = rsp[mask]

    if len(rsp_clip) == 0:
        print("⚠ No respiration data in selected window. Skipping.")
        continue

    # --------------------------------------------------
    # FIGURE SETUP
    # --------------------------------------------------

    fig, (ax_vid, ax_rsp) = plt.subplots(
        2, 1,
        figsize=(7, 7),
        gridspec_kw={"height_ratios": [3, 1]}
    )

    ret, frame = cap.read()
    if not ret:
        print("⚠ Could not read first frame. Skipping.")
        plt.close(fig)
        continue

    im = ax_vid.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax_vid.set_title(base_name)
    ax_vid.axis("off")

    line, = ax_rsp.plot([], [], color="black", lw=1)
    ax_rsp.set_ylim(
        np.percentile(rsp_clip, 1),
        np.percentile(rsp_clip, 99)
    )
    ax_rsp.set_xlabel("Time (s)")
    ax_rsp.set_ylabel("Respiration")

    # --------------------------------------------------
    # UPDATE FUNCTION
    # --------------------------------------------------

    def update(frame_idx):

        actual_frame = start_frame + frame_idx
        if actual_frame >= end_frame:
            return im, line

        cap.set(cv2.CAP_PROP_POS_FRAMES, actual_frame)
        ret, frame = cap.read()
        if not ret:
            return im, line

        im.set_data(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        current_time = frame_idx / fps

        window_mask = (
            (t_clip >= current_time - WINDOW) &
            (t_clip <= current_time)
        )

        line.set_data(t_clip[window_mask], rsp_clip[window_mask])
        ax_rsp.set_xlim(max(0, current_time - WINDOW), current_time)

        return im, line

    # --------------------------------------------------
    # CREATE ANIMATION
    # --------------------------------------------------

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=(end_frame - start_frame),
        interval=1000 / fps,
        blit=False
    )

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------

    clip_label = f"{CLIP_START}-{CLIP_END}s"
    save_filename = f"{base_name}_{clip_label}_resp_overlay.mp4"
    save_path = os.path.join(qc_folder, save_filename)

    print("Saving to:", save_path)

    ani.save(save_path, writer="ffmpeg", fps=fps)

    print("Saved successfully.")

    plt.close(fig)


Total H5 files found: 97

Processing: 1_1_RI1_f1_d_on_20260111_164638
Saving to: D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2\1_1_RI1_f1_d_on_20260111_164638_60-240s_resp_overlay.mp4


C:\Users\sjs93\AppData\Local\Temp\ipykernel_26140\1514015740.py:141: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax_rsp.set_xlim(max(0, current_time - WINDOW), current_time)


Saved successfully.

Processing: 1_2_RI1_f2_f_off_20260109_165330
Saving to: D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2\1_2_RI1_f2_f_off_20260109_165330_60-240s_resp_overlay.mp4


C:\Users\sjs93\AppData\Local\Temp\ipykernel_26140\1514015740.py:141: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax_rsp.set_xlim(max(0, current_time - WINDOW), current_time)


Saved successfully.

Processing: 1_3_RI1_f1_c_off_20260110_130254
Saving to: D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2\1_3_RI1_f1_c_off_20260110_130254_60-240s_resp_overlay.mp4


...

Saved successfully.

Processing: 1_1_RI1_f1_d_on_20260111_164638
Saving to: D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2\1_1_RI1_f1_d_on_20260111_164638_60-240s_resp_overlay.mp4


C:\Users\sjs93\AppData\Local\Temp\ipykernel_26140\1514015740.py:141: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax_rsp.set_xlim(max(0, current_time - WINDOW), current_time)


Saved successfully.

Processing: 1_1_RI2_f2_h_off_20260119_134817
Saving to: D:\BLA_ChR_resp\QC_fulltrace_plots_RI1_2\1_1_RI2_f2_h_off_20260119_134817_60-240s_resp_overlay.mp4


C:\Users\sjs93\AppData\Local\Temp\ipykernel_26140\1514015740.py:141: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax_rsp.set_xlim(max(0, current_time - WINDOW), current_time)


Saved successfully.
